# 20261103 - Intro to Accelerators (e.g. GPUs)

## Computing Hardware Manufacturing Background

- Lots of constraints go into making of silicon integrated circuits
- Biggest amongst them is area
    - Silicon fabrication companies ("fabs") charge by the wafer
    - Wafer has set size
    - Manufacturing can only fit so many chips onto a single wafer

- Can't "just have more ____"
    - Adding to one takes away from something else
    - Chiplet/multi-die design alleviates this somewhat, but you will then run into physics (speed of light latency)
   

- Not all CPU components take up the same space
    - A single logic transitor uses at least 2 actual transistors (e.g. a CMOS)
    - A single bit of SRAM (used for cache) requires 4-10 transistors
    - Additionally, SRAM can't use as small of transistors as logic
        - Put simply, SRAM needs to *stably* store energy, while logic simply needs to switch
- Thus, its not even a one-for-one trade between logic vs cache

Modern CPUs have:
- Cache hierarchies
- Pipelining and multiple issue
- Branch predicition
- Super-scalar architectures (multiple execution units)
- Out-of-order execution

Their "die budget" is broken down like

| Item | Die Area (%) |
| --- | --- |
| L3 Cache | 18.8 |
| L2 Cache | 6.17 |
| L1 Cache | 1.44 |
| ALU (integer and floating point) | 2.82 |
| Load/Store | 2.46 |
| Execution control logic | 7.26 |
| Misc Connection logic | 15.94 |
| Misc I/O handling | 46.8 |

**Why do we put so little in ALU and so much in everything else?**

Because CPUs are designed for complicated logic execution, with many unknowns
- We don't know what memory we need next, so we need caches
- We don't know what instructions are coming up, so we need pipelining and branch prediction
- We don't know what kind of instructions we'll run, so we need lots of different execution units

**But what if we did know and could specialize?**

## CPU-Accelerator Offloading

- We still want to run complicated, poorly predicted workloads well
    - e.g. we still want flexibility in our compute
- But there are many operations that are *very* predictable
- So predictable, that they can be run with an entirely different hardware design
- What if we just had a dedicated hardware chip handle the super predictable work?

The idea:
- CPU is still the "master" device
    - It dictates who does what
- The CPU offloads tasks to the accelerator
- The accelerator communicates results back to CPU

### Examples
- GPUs (duh)

- Network Interface Cards
    - Beyond the analog to digital hardware, CPUs *could* perform the exact same function
    - But a dedicated NIC is significantly more efficient
- Sound cards
    - CPUs *could* do all of the digital signal processing necessary
    - But dedciated sound cards are *way* more efficient at it
    - A cheap $20 external sound card can easily outperform a much more expensive CPU (and GPU for that matter)
- (Sometime in the future) Quantum Computing
    - Modern computers can simulate quantum systems
    - Actual quantum systems are much more efficient

### Devil in the details

- Sounds ideal! Let everyone do what their best at!
- But reality (and physics) can ruin things

- Latency: How long will it take for CPU to get results back?
    - Does that latency outweigh the performance gains?

- Code complexity: How easy is it to offload tasks to accelerator?
    - Can I just tell the accelerator "run this CPU task"?
    - Do I need to translate the task into a form that the accelerator can work with?
        - Just a new language?
        - Or do I need an entirely new algorithm?

- Complexities of the accelerator itself
    - How much work can it take?
    - Does it have enough memory to hold the entire problem?
    - How often does it need the CPU to intervene?

### Why not just tweak the CPU?

We have a spectrum of maximally flexible (CPU) to maximally performant (accelerator/ASIC).

**Why not have something in the middle?**

Short answer: economics
- A much larger market working in the extremes
    - Need flexibility? Get the best, most flexible thing out there
    - Need max performance? Get the best performing hardware
- You can get something "kinda" in between by just having the CPU+accelerator offloading system

#### Noteworthy Exceptions
**Mainframes**
- These are (IBM) machines which are built for max uptime, but for very specific workloads (namely real-time financial processing)
- They're still flexibly programmable, but with bespoke OS, compilers, and architecture (IBM's Z architecture)
    - Though the latest IBM mainframes can [run ARM and Z instructions on the same CPU](https://newsroom.ibm.com/2026-08-24-ibm-unveils-next-generation-dual-architecture-processor-for-ibm-z-and-linuxone)
   

**ARM CPUs (e.g. the future ideal for scientific workloads IMO)**
- ARM is a licensed instruction set architecture
- Also provides pre-designed core designs
- Allows for gluing together custom designed chips to orient towards specific workloads
- This powers the fastest machine in the world
    - LineShine (China) is an ARM based machine
    - Fugaku (Japan), previous #1, also ARM based machine
    - Both use custom ARM chips with more die area towards vector instructions
- Fugaku chip has a heterogenous CPU architecture
    - Has 48 vector-heavy CPU cores, and 2 "assistant" cores for handling overhead
    

## Caution When Comparing Hardware

- Comparing "speed" and "performance" between different hardware is hard
- You can't just say "I went 20x faster on GPUs"
    - This is a non-sense statement without further context
    - Equivalent of saying "I was 20x faster on a train than driving"
        - Driving what? Around where? How far? What train were you on?

**Need to constrain some factors before doing a comparison**
- General comparison
    - Performance per watt
    - Performance per $
- Situational comparison
    - Performance per node
    - Performance per throughput on Supercomputing machine
        - More nodes maybe faster, but wait longer in queue
        - Especially true if GPU/accelerators are limited
    - Performance per allocation cost metric
        - Supercomputer allocation costs don't necessarily line up with hardware or energy cost

### Still not always a fair race (depending on your comparison goal)
- We've shown memory bandwidth can be a hugebottle neck
- GPUs always have on package, high bandwidth memory
    - (This is a requirement for their architecture, we'll talk about later)
- CPUs don't normally have this
- Thus, in some cases, it's less "GPUs are faster than CPUs" and more "GPUs have faster memory than CPUs"
- Some CPUs now have HBM

## GPUs - A short history

- GPUs pre-2000 were *very* specialized hardware, basically a modern ASIC (application specific integrated circuit)
    - Near zero execution control logic
    - Hardwired functionality, fixed pipeline
        - "take triangle, take 64x64 texture, apply to triangle, run specific lighting algorithm"
        - Want to use a 128x128 texture? Apply to a rectangle? Have a new lighting algorithm? Too bad, can't do it
       

- GPUs then got "programmable shaders" in early 2000s
    - You can run the pipeline in different orders
    - Have some control over specific parts of the pipeline
    - Still a collection of fixed function hardware
        - Want to devote more resources to lighting than textures? Too bad, can't do it

- NVIDIA introduced "unified shaders" in 2006
    - Also called Streaming Multiprocessor (SM, NVIDIA), Compute Units (AMD), or a XeCore (Intel)
    - A collection of general purpose math processors that could be programmed arbitrarily
    - Graphics engines could now dictate which operations to devote more hardware towards